# DiffDock-Pocket 虚拟筛选教程

**论文引用:** Plainer, M., Toth, M., Dober, S., Stark, H., & Gunnemann, S. (2023). *DiffDock-Pocket: Diffusion for Pocket-Level Docking with Sidechain Flexibility.* NeurIPS 2023 Workshop on Generative AI and Biology.

## 核心创新: 口袋柔性扩散 (Pocket-Flexible Diffusion)

DiffDock-Pocket 在 DiffDock 的 **3-way 扩散** (配体平移 + 旋转 + 扭转角) 基础上，
新增了 **第 4 维扩散 -- 蛋白侧链 chi1 扭转角**，实现柔性受体对接:

| 方法 | 扩散维度 | 描述 |
|------|----------|------|
| DiffDock | 3-way | 配体 translation + rotation + torsion |
| **DiffDock-Pocket** | **4-way** | 配体 tr + rot + tor + **sidechain chi** |

关键技术要点:
- 蛋白采用 **残基级别** 表示 (CA/CB 坐标 + 残基类型独热编码)
- 通过学习侧链 chi1 角度的分数函数，实现对接过程中蛋白口袋的柔性调整
- 参考原始代码: `DiffDock-Pocket/utils/diffusion_utils.py` 中的 `t_to_sigma` 函数使用 4 个独立的 sigma 通道

## 学习目标

1. 理解 DiffDock-Pocket 在 DiffDock 基础上新增的侧链扭转角扩散维度
2. 掌握残基级别蛋白表示与原子级别配体表示之间的交互建模
3. 学习 4-way 分数匹配 (score matching) 训练策略
4. 实践反向扩散对接的推理流程，同时优化配体位姿和蛋白口袋构象

In [ ]:
# ============================================================
# 环境设置、路径配置与依赖导入
# ============================================================

def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    from pathlib import Path
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'数据目录:   {DATA_DIR}')

import os
import math
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit import RDLogger
from scipy.spatial import distance_matrix

%matplotlib inline
import matplotlib.pyplot as plt

# BioPython 用于残基级别的蛋白解析
from Bio.PDB import PDBParser

# 抑制 RDKit / BioPython 的冗余警告
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')

# 版本信息
version_info = pd.DataFrame({
    '依赖库': ['torch', 'rdkit', 'numpy', 'scipy', 'biopython', 'pandas'],
    '版本': [
        torch.__version__,
        Chem.rdBase.rdkitVersion,
        np.__version__,
        __import__('scipy').__version__,
        __import__('Bio').__version__,
        pd.__version__,
    ]
})
display(version_info)

## 1. 超参数设置

DiffDock-Pocket 的超参数分为两组:
- **模型超参数**: 特征维度、隐层维度、交互层数、距离阈值等
- **扩散噪声参数**: 对应原始代码中的 `sigma_min` / `sigma_max`，控制 4 个扩散维度的噪声强度

其中侧链 chi1 噪声参数 `SIGMA_SC_MAX` 是 DiffDock-Pocket 相比 DiffDock 新增的关键超参数。

In [ ]:
# ============================================================
# 超参数定义
# ============================================================

# ---------- 模型超参数 ----------
ATOM_FEAT_DIM = 10          # 配体原子特征维度
RES_FEAT_DIM = 21           # 残基特征维度 (20 种标准氨基酸 + 1 other)
HIDDEN_DIM = 128            # 隐层维度
N_INTERACTION_LAYERS = 4    # 蛋白-配体交互层数
DISTANCE_CUTOFF = 8.0       # 残基-原子交互距离阈值 (Angstrom)
N_EPOCHS = 200              # 训练轮数
LR = 1e-4                   # 学习率 (扩散模型使用较小学习率)
BATCH_SIZE = 1              # 变长图，逐样本处理
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- 扩散噪声参数 (对应原始代码的 sigma_min / sigma_max) ----------
SIGMA_TR_MAX = 10.0         # 平移噪声最大标准差 (Angstrom)
SIGMA_ROT_MAX = 1.5         # 旋转噪声最大标准差 (弧度)
SIGMA_TOR_MAX = np.pi       # 配体扭转角噪声最大标准差
SIGMA_SC_MAX = np.pi        # 侧链 chi1 噪声最大标准差 (弧度) -- DiffDock-Pocket 新增!

torch.manual_seed(SEED)
np.random.seed(SEED)

# 展示超参数
params_df = pd.DataFrame({
    '参数': ['ATOM_FEAT_DIM', 'RES_FEAT_DIM', 'HIDDEN_DIM', 'N_INTERACTION_LAYERS',
            'DISTANCE_CUTOFF', 'N_EPOCHS', 'LR', 'BATCH_SIZE',
            'SIGMA_TR_MAX', 'SIGMA_ROT_MAX', 'SIGMA_TOR_MAX', 'SIGMA_SC_MAX'],
    '值': [ATOM_FEAT_DIM, RES_FEAT_DIM, HIDDEN_DIM, N_INTERACTION_LAYERS,
           DISTANCE_CUTOFF, N_EPOCHS, LR, BATCH_SIZE,
           SIGMA_TR_MAX, SIGMA_ROT_MAX, f'{SIGMA_TOR_MAX:.4f}', f'{SIGMA_SC_MAX:.4f}'],
    '说明': ['配体原子特征维度', '残基特征维度 (20+1)', '隐层维度', '蛋白-配体交互层数',
            '交互距离阈值 (A)', '训练轮数', '学习率', '批大小 (变长图逐样本)',
            '平移噪声最大标准差 (A)', '旋转噪声最大标准差 (rad)', '扭转角噪声最大标准差',
            '侧链 chi1 噪声最大标准差 (新增!)']
})
display(params_df)

print(f"\nDevice: {DEVICE}")
print(f"DiffDock-Pocket: 4-way diffusion (tr + rot + tor + sidechain chi)")

## 2. 数据加载与特征提取

本节定义以下功能:

1. **CoreSet.dat 解析** -- 读取 PDBbind CASF-2016 核心集的结合亲和力标签
2. **配体原子特征提取** -- 10 维向量 (元素类型 one-hot + 芳香性)
3. **蛋白残基特征提取** -- 21 维向量 (氨基酸类型 one-hot)，使用 BioPython 解析 PDB 获取 CA/CB 坐标和侧链原子
4. **侧链 chi1 角度计算** -- 用 CA->CB 方向向量在 xy 平面上的投影角作为代理
5. **侧链扭转角修改** -- Rodrigues 旋转公式绕 CA->CB 轴旋转侧链原子
6. **扩散工具函数** -- 4-way 噪声调度、配体加噪、侧链加噪
7. **正弦位置编码** -- 将扩散时间步映射为高维嵌入向量

In [ ]:
# ============================================================
# 特征提取与扩散工具函数
# ============================================================

# ---- 标准氨基酸列表 ----
STANDARD_AAS = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
                'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL']

# ---- 元素符号 -> one-hot 映射 (9类 + 1 芳香性 = 10维) ----
ELEMENT_LIST = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br']


def parse_coreset(path):
    """解析 CoreSet.dat，返回 {pdbid: logKa} 字典。跳过以 '#' 开头的注释行。"""
    labels = {}
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            pdbid = parts[0]
            logKa = float(parts[3])
            labels[pdbid] = logKa
    print(f"[INFO] 从 CoreSet.dat 读取到 {len(labels)} 个复合物标签")
    return labels


def atom_features(atom):
    """
    构建 10 维配体原子特征向量:
      - 前 9 维: 元素类型 one-hot (C, N, O, S, F, P, Cl, Br, other)
      - 第 10 维: 是否为芳香性原子
    """
    feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
    symbol = atom.GetSymbol()
    if symbol in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(symbol)] = 1.0
    else:
        feat[8] = 1.0
    feat[9] = float(atom.GetIsAromatic())
    return feat


def residue_features(resname):
    """
    构建 21 维残基特征向量:
      - 前 20 维: 标准氨基酸类型 one-hot
      - 第 21 维: 非标准氨基酸 (other)
    """
    feat = np.zeros(RES_FEAT_DIM, dtype=np.float32)
    if resname in STANDARD_AAS:
        feat[STANDARD_AAS.index(resname)] = 1.0
    else:
        feat[20] = 1.0
    return feat


def extract_residue_data(pocket_pdb):
    """
    从口袋 PDB 文件中提取残基级别信息 (用 BioPython PDBParser):
      - res_feats  : (N_r, 21)  残基类型特征
      - ca_coords  : (N_r, 3)   CA 原子坐标
      - cb_coords  : (N_r, 3)   CB 原子坐标 (Gly 无 CB 时用 CA 代替)
      - sc_atoms   : list        每个残基的侧链原子坐标列表
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pocket', pocket_pdb)

    res_feats_list = []
    ca_coords_list = []
    cb_coords_list = []
    sc_atoms_list = []

    # 主链原子名 (排除这些以获得侧链原子)
    backbone_atoms = {'N', 'CA', 'C', 'O', 'OXT'}

    for model in structure:
        for chain in model:
            for residue in chain:
                # 跳过水分子和 HETATM
                if residue.id[0] != ' ':
                    continue
                resname = residue.get_resname().strip()

                # 获取 CA 坐标
                if 'CA' not in residue:
                    continue
                ca_coord = residue['CA'].get_vector().get_array()

                # 获取 CB 坐标 (Gly 没有 CB，用 CA 代替)
                if 'CB' in residue:
                    cb_coord = residue['CB'].get_vector().get_array()
                else:
                    cb_coord = ca_coord.copy()

                # 收集侧链原子坐标 (排除主链原子)
                sc_coords = []
                for atom in residue:
                    if atom.get_name() not in backbone_atoms:
                        sc_coords.append(atom.get_vector().get_array())

                res_feats_list.append(residue_features(resname))
                ca_coords_list.append(ca_coord)
                cb_coords_list.append(cb_coord)
                sc_atoms_list.append(
                    np.array(sc_coords, dtype=np.float32) if len(sc_coords) > 0
                    else np.zeros((0, 3), dtype=np.float32)
                )
        break  # 只取第一个 model

    if len(ca_coords_list) == 0:
        return None

    res_feats = np.array(res_feats_list, dtype=np.float32)
    ca_coords = np.array(ca_coords_list, dtype=np.float32)
    cb_coords = np.array(cb_coords_list, dtype=np.float32)
    return res_feats, ca_coords, cb_coords, sc_atoms_list


def extract_chi1_angles(ca_coords, cb_coords):
    """
    计算简化的 chi1 角度代理值。
    用 CA->CB 方向向量在 xy 平面上的投影角作为 chi1 的代理。
    对于 Gly (CB == CA)，chi1 设为 0。

    返回: chi1_angles (N_r,) 弧度制
    """
    diff = cb_coords - ca_coords                     # (N_r, 3)
    norms = np.linalg.norm(diff, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-6, None)               # 避免除零
    direction = diff / norms
    chi1 = np.arctan2(direction[:, 1], direction[:, 0]).astype(np.float32)
    return chi1


def modify_sidechain_torsion(sc_atoms, ca_coord, cb_coord, chi1_update):
    """
    通过绕 CA->CB 轴旋转来模拟侧链扭转角的变化 (Rodrigues 旋转公式)。

    参数:
      sc_atoms    : (N_sc, 3)  侧链原子坐标
      ca_coord    : (3,)       CA 原子坐标 (旋转轴起点)
      cb_coord    : (3,)       CB 原子坐标 (旋转轴方向)
      chi1_update : float      旋转角度 (弧度)

    返回: new_sc_atoms : (N_sc, 3)
    """
    if len(sc_atoms) == 0:
        return sc_atoms.copy()

    axis = cb_coord - ca_coord
    axis_norm = np.linalg.norm(axis)
    if axis_norm < 1e-6:
        return sc_atoms.copy()          # Gly 无侧链轴
    axis = axis / axis_norm

    cos_a = np.cos(chi1_update)
    sin_a = np.sin(chi1_update)
    centered = sc_atoms - ca_coord
    cross = np.cross(axis, centered)
    dot = np.dot(centered, axis).reshape(-1, 1)
    rotated = centered * cos_a + cross * sin_a + axis * dot * (1 - cos_a)
    return rotated + ca_coord


def load_ligand(pdbid):
    """加载配体分子，返回 (原子特征, 3D 坐标)。优先 mol2，备选 sdf。"""
    ligand_mol2 = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_ligand.mol2")
    ligand_sdf = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_ligand.sdf")

    lig_mol = Chem.MolFromMol2File(ligand_mol2, sanitize=False)
    if lig_mol is None and os.path.exists(ligand_sdf):
        supplier = Chem.SDMolSupplier(ligand_sdf, sanitize=False)
        for m in supplier:
            if m is not None:
                lig_mol = m
                break
    if lig_mol is None:
        return None

    try:
        Chem.SanitizeMol(lig_mol)
    except Exception:
        pass
    try:
        lig_mol = Chem.RemoveHs(lig_mol)
    except Exception:
        pass
    if lig_mol.GetNumAtoms() == 0:
        return None

    conf = lig_mol.GetConformer()
    coords = np.array([conf.GetAtomPosition(i) for i in range(lig_mol.GetNumAtoms())],
                       dtype=np.float32)
    feats = np.array([atom_features(a) for a in lig_mol.GetAtoms()], dtype=np.float32)
    return feats, coords


# ---- 正弦位置编码 (扩散时间步嵌入) ----
class SinusoidalEmbedding(nn.Module):
    """
    正弦/余弦位置编码，将标量时间步 t 映射到高维向量。
    对应原始代码 diffusion_utils.py 中的 sinusoidal_embedding 函数。
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        """t: (B,) -> (B, dim)"""
        if t.dim() == 0:
            t = t.unsqueeze(0)
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000.0)
            * torch.arange(half, device=t.device, dtype=torch.float32) / half
        )
        args = t.unsqueeze(-1) * freqs.unsqueeze(0)
        return torch.cat([args.sin(), args.cos()], dim=-1)


# ---- 4-way 噪声调度 ----
def t_to_sigma_4way(t):
    """
    将归一化时间 t in [0,1] 映射到 4 个独立的噪声标准差。

    DiffDock-Pocket 的核心创新: 在 DiffDock 的 3-way 基础上新增 sc_tor_sigma。

    返回: (sigma_tr, sigma_rot, sigma_tor, sigma_sc_chi)
    """
    # log-linear (exponential) 调度: sigma(t) = sigma_max^t
    sigma_tr = SIGMA_TR_MAX ** t
    sigma_rot = SIGMA_ROT_MAX ** t
    sigma_tor = SIGMA_TOR_MAX ** t
    sigma_sc = SIGMA_SC_MAX ** t
    return sigma_tr, sigma_rot, sigma_tor, sigma_sc


def apply_noise_to_ligand(lig_coords, t):
    """
    对配体坐标施加 3-way 噪声 (平移 + 旋转 + 扭转)。
    这里简化为教学版本。

    返回: (noisy_coords, noise_tr, noise_rot, noise_tor)
    """
    sigma_tr, sigma_rot, sigma_tor, _ = t_to_sigma_4way(t)

    # ---- 1. 平移噪声 ----
    noise_tr = np.random.randn(3).astype(np.float32) * sigma_tr
    noisy = lig_coords + noise_tr

    # ---- 2. 旋转噪声 (绕质心随机旋转) ----
    center = noisy.mean(axis=0)
    axis = np.random.randn(3).astype(np.float32)
    axis_norm = np.linalg.norm(axis)
    if axis_norm > 1e-6:
        axis = axis / axis_norm
    else:
        axis = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    angle = np.random.randn() * sigma_rot
    noise_rot = axis * angle  # 轴角表示 (3,)

    cos_a, sin_a = np.cos(angle), np.sin(angle)
    centered = noisy - center
    cross = np.cross(axis, centered)
    dot = np.dot(centered, axis).reshape(-1, 1)
    noisy = centered * cos_a + cross * sin_a + axis * dot * (1 - cos_a) + center

    # ---- 3. 扭转角噪声 (简化: 径向扰动) ----
    noise_tor_val = np.random.randn() * sigma_tor
    radial = noisy - center
    radial_norms = np.linalg.norm(radial, axis=1, keepdims=True)
    radial_norms = np.clip(radial_norms, 1e-6, None)
    noisy = noisy + (radial / radial_norms) * noise_tor_val * 0.1

    return noisy, noise_tr, noise_rot, np.float32(noise_tor_val)


def apply_noise_to_sidechain(chi1_angles, t):
    """
    对蛋白侧链 chi1 角施加噪声。
    这是 DiffDock-Pocket 相比 DiffDock 的核心新增!

    返回: (noisy_chi1, noise_sc)
    """
    _, _, _, sigma_sc = t_to_sigma_4way(t)
    noise_sc = np.random.randn(len(chi1_angles)).astype(np.float32) * sigma_sc
    noisy_chi1 = chi1_angles + noise_sc
    # 归一化到 [-pi, pi]
    noisy_chi1 = np.arctan2(np.sin(noisy_chi1), np.cos(noisy_chi1))
    return noisy_chi1, noise_sc


def compute_rmsd(pred, true):
    """计算 RMSD (均方根偏差)。"""
    return np.sqrt(np.mean(np.sum((pred - true) ** 2, axis=1)))

In [ ]:
# ============================================================
# 展示一个样本的数据加载结果
# ============================================================

# 解析标签
labels = parse_coreset(CORESET_FILE)

# 取第一个 pdbid 作为示例
example_pdbid = sorted(labels.keys())[0]
print(f"\n示例 PDB ID: {example_pdbid}, logKa = {labels[example_pdbid]:.2f}")

# 加载配体
lig_result = load_ligand(example_pdbid)
if lig_result is not None:
    lig_feats_ex, lig_coords_ex = lig_result
    print(f"配体原子特征形状: {lig_feats_ex.shape}  (N_atoms, {ATOM_FEAT_DIM})")
    print(f"配体坐标形状:     {lig_coords_ex.shape}  (N_atoms, 3)")

# 加载蛋白口袋
pocket_path = os.path.join(COMPLEX_DIR, example_pdbid, f"{example_pdbid}_pocket.pdb")
res_result = extract_residue_data(pocket_path)
if res_result is not None:
    res_feats_ex, ca_coords_ex, cb_coords_ex, sc_atoms_ex = res_result
    chi1_ex = extract_chi1_angles(ca_coords_ex, cb_coords_ex)
    print(f"残基特征形状:     {res_feats_ex.shape}  (N_residues, {RES_FEAT_DIM})")
    print(f"CA 坐标形状:      {ca_coords_ex.shape}  (N_residues, 3)")
    print(f"CB 坐标形状:      {cb_coords_ex.shape}  (N_residues, 3)")
    print(f"Chi1 角度形状:    {chi1_ex.shape}  (N_residues,)")
    print(f"侧链原子组数:    {len(sc_atoms_ex)}")

    sample_df = pd.DataFrame({
        '数据': ['配体原子数', '口袋残基数', '配体特征维度', '残基特征维度', 'Chi1 角度范围'],
        '值': [lig_feats_ex.shape[0], res_feats_ex.shape[0],
               ATOM_FEAT_DIM, RES_FEAT_DIM,
               f'[{np.degrees(chi1_ex.min()):.1f}, {np.degrees(chi1_ex.max()):.1f}] deg']
    })
    display(sample_df)

## 3. 数据集与数据加载器

构建 PyTorch Dataset 和 DataLoader:
- 每个样本包含: 配体原子特征/坐标、蛋白残基特征/CA/CB 坐标、侧链 chi1 角度、PDB ID 和标签
- 由于不同复合物的原子/残基数量不同 (变长图)，使用 `batch_size=1` 并自定义 `collate_fn`
- 按 80/20 比例随机划分训练/测试集

In [ ]:
# ============================================================
# Dataset 与 DataLoader
# ============================================================

class DiffDockPocketDataset(Dataset):
    """
    DiffDock-Pocket 数据集。每个样本包含:
      - 配体原子特征与坐标 (真实位姿)
      - 蛋白残基特征、CA/CB 坐标
      - 侧链 chi1 角度
      - PDB ID
    """
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def build_complex_data(pdbid):
    """
    为单个蛋白-配体复合物构建数据。
    返回字典或 None (解析失败时)。
    """
    pocket_path = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_pocket.pdb")

    # ---- 加载配体 ----
    lig_result = load_ligand(pdbid)
    if lig_result is None:
        return None
    lig_feats, lig_coords = lig_result

    # ---- 加载蛋白口袋 (残基级别) ----
    if not os.path.exists(pocket_path):
        return None
    res_result = extract_residue_data(pocket_path)
    if res_result is None:
        return None
    res_feats, ca_coords, cb_coords, sc_atoms = res_result

    if len(ca_coords) == 0:
        return None

    # ---- 计算 chi1 角度 ----
    chi1_angles = extract_chi1_angles(ca_coords, cb_coords)

    return {
        'lig_feats': lig_feats,       # (N_l, 10)
        'lig_coords': lig_coords,     # (N_l, 3)  真实配体位姿
        'res_feats': res_feats,       # (N_r, 21)
        'ca_coords': ca_coords,       # (N_r, 3)
        'cb_coords': cb_coords,       # (N_r, 3)
        'chi1_angles': chi1_angles,   # (N_r,)
        'sc_atoms': sc_atoms,         # list of (N_sc_i, 3)
    }


# ---- 预处理所有复合物 ----
print("\n正在构建 DiffDock-Pocket 数据...")

all_data = []
failed = 0
for pdbid, logKa in sorted(labels.items()):
    result = build_complex_data(pdbid)
    if result is None:
        failed += 1
        continue
    result['label'] = logKa
    result['pdbid'] = pdbid
    all_data.append(result)

print(f"成功加载 {len(all_data)} 个复合物, 失败 {failed} 个")

# ---- 随机 80/20 划分训练/测试集 ----
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_idx, test_idx = indices[:split], indices[split:]

train_data = [all_data[i] for i in train_idx]
test_data = [all_data[i] for i in test_idx]

train_loader = DataLoader(DiffDockPocketDataset(train_data), batch_size=BATCH_SIZE,
                           shuffle=True, collate_fn=lambda x: x[0])
test_loader = DataLoader(DiffDockPocketDataset(test_data), batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=lambda x: x[0])

# 展示数据集划分信息
split_df = pd.DataFrame({
    '数据集': ['训练集', '测试集', '总计'],
    '样本数': [len(train_data), len(test_data), len(all_data)],
    '比例': [f'{len(train_data)/len(all_data)*100:.1f}%',
            f'{len(test_data)/len(all_data)*100:.1f}%',
            '100%']
})
display(split_df)

## 4. 模型架构

DiffDock-Pocket 的分数模型由以下组件构成:

1. **时间步嵌入** -- 正弦位置编码将标量 t 映射到 128 维向量
2. **蛋白残基嵌入** -- 21 维残基特征映射到 128 维隐表示
3. **配体原子嵌入** -- 10 维原子特征映射到 128 维隐表示
4. **交互层 x 4** -- 双向消息传递: 残基级别蛋白与原子级别配体之间的信息交换
5. **4 个输出头**:
   - `tr_head` -- 平移分数 (3 维)
   - `rot_head` -- 旋转分数 (3 维)
   - `tor_head` -- 扭转角分数 (1 维)
   - `sc_tor_head` -- **侧链 chi1 扭转分数** (每残基 1 维) -- **DiffDock-Pocket 新增!**

与 DiffDock 相比，唯一的架构差异就是新增了 `sc_tor_head`，为每个蛋白残基预测一个侧链 chi1 扭转分数。

In [ ]:
# ============================================================
# 模型架构 -- ToyDiffDockPocketScore
# ============================================================

class InteractionLayer(nn.Module):
    """
    蛋白-配体交互层: 双向消息传递。
    残基级别蛋白表示与原子级别配体表示之间的信息交换。
    """
    def __init__(self, hidden_dim):
        super().__init__()
        # 蛋白残基 -> 配体原子
        self.prot_to_lig = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        # 配体原子 -> 蛋白残基
        self.lig_to_prot = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        # 节点更新 MLP
        self.prot_update = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.lig_update = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.layer_norm_prot = nn.LayerNorm(hidden_dim)
        self.layer_norm_lig = nn.LayerNorm(hidden_dim)

    def forward(self, prot_h, lig_h, edge_index, edge_dist):
        """
        prot_h     : (N_r, hidden)   lig_h      : (N_l, hidden)
        edge_index : (2, E) [0]=lig   edge_dist  : (E, 1)
        """
        lig_idx = edge_index[0]
        prot_idx = edge_index[1]

        # ---- 蛋白 -> 配体 消息 ----
        msg_p2l = self.prot_to_lig(
            torch.cat([lig_h[lig_idx], prot_h[prot_idx], edge_dist], dim=-1)
        )
        lig_agg = torch.zeros_like(lig_h)
        count_l = torch.zeros(lig_h.shape[0], 1, device=lig_h.device)
        lig_agg.scatter_add_(0, lig_idx.unsqueeze(-1).expand_as(msg_p2l), msg_p2l)
        count_l.scatter_add_(0, lig_idx.unsqueeze(-1), torch.ones_like(edge_dist))
        lig_agg = lig_agg / count_l.clamp(min=1)

        # ---- 配体 -> 蛋白 消息 ----
        msg_l2p = self.lig_to_prot(
            torch.cat([prot_h[prot_idx], lig_h[lig_idx], edge_dist], dim=-1)
        )
        prot_agg = torch.zeros_like(prot_h)
        count_p = torch.zeros(prot_h.shape[0], 1, device=prot_h.device)
        prot_agg.scatter_add_(0, prot_idx.unsqueeze(-1).expand_as(msg_l2p), msg_l2p)
        count_p.scatter_add_(0, prot_idx.unsqueeze(-1), torch.ones_like(edge_dist))
        prot_agg = prot_agg / count_p.clamp(min=1)

        # ---- 残差更新 ----
        lig_h = self.layer_norm_lig(
            lig_h + self.lig_update(torch.cat([lig_h, lig_agg], dim=-1))
        )
        prot_h = self.layer_norm_prot(
            prot_h + self.prot_update(torch.cat([prot_h, prot_agg], dim=-1))
        )
        return prot_h, lig_h


class ToyDiffDockPocketScore(nn.Module):
    """
    DiffDock-Pocket 分数模型。

    与 DiffDock 相比，新增了 sc_tor_head -- 这是唯一的架构差异:
      DiffDock:        tr_head + rot_head + tor_head           (3 个头)
      DiffDock-Pocket: tr_head + rot_head + tor_head + sc_tor_head (4 个头)

    sc_tor_head 为每个蛋白残基预测一个侧链 chi1 扭转分数，
    使得扩散过程能同时优化配体位姿和蛋白口袋构象。
    """
    def __init__(self, lig_dim=ATOM_FEAT_DIM, res_dim=RES_FEAT_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()

        # ---- 时间步嵌入 ----
        self.time_embed = SinusoidalEmbedding(hidden_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # ---- 蛋白残基嵌入 (残基级别: 21 -> 128) ----
        self.prot_embed = nn.Linear(res_dim, hidden_dim)

        # ---- 配体原子嵌入 (原子级别: 10 -> 128) ----
        self.lig_embed = nn.Linear(lig_dim, hidden_dim)

        # ---- 交互层 x 4 ----
        self.interaction_layers = nn.ModuleList([
            InteractionLayer(hidden_dim) for _ in range(N_INTERACTION_LAYERS)
        ])

        # ---- DiffDock 的 3 个输出头 ----
        self.tr_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 3),
        )
        self.rot_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 3),
        )
        self.tor_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

        # ---- DiffDock-Pocket 新增: 侧链扭转角分数头 ----
        self.sc_tor_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, lig_feats, res_feats, edge_index, edge_dist, t):
        """
        参数:
          lig_feats  : (N_l, 10)   配体原子特征
          res_feats  : (N_r, 21)   蛋白残基特征
          edge_index : (2, E)      交互边 [0]=配体, [1]=蛋白
          edge_dist  : (E, 1)      边距离
          t          : (1,)        归一化时间步

        返回:
          score_tr  : (3,)       score_rot : (3,)
          score_tor : (1,)       score_sc  : (N_r, 1)  <-- NEW
        """
        # ---- 时间步条件 ----
        time_h = self.time_mlp(self.time_embed(t))   # (1, hidden)

        # ---- 节点嵌入 + 时间步注入 ----
        lig_h = self.lig_embed(lig_feats) + time_h
        prot_h = self.prot_embed(res_feats) + time_h

        # ---- 4 层交互 ----
        for layer in self.interaction_layers:
            prot_h, lig_h = layer(prot_h, lig_h, edge_index, edge_dist)

        # ---- 全局聚合 (mean + max) ----
        lig_mean = lig_h.mean(dim=0, keepdim=True)
        lig_max = lig_h.max(dim=0, keepdim=True).values
        lig_global = torch.cat([lig_mean, lig_max], dim=-1)

        prot_mean = prot_h.mean(dim=0, keepdim=True)
        prot_max = prot_h.max(dim=0, keepdim=True).values
        prot_global = torch.cat([prot_mean, prot_max], dim=-1)

        # ---- 3 个配体分数 (与 DiffDock 相同) ----
        joint_global = lig_global + prot_global
        score_tr = self.tr_head(joint_global).squeeze(0)
        score_rot = self.rot_head(joint_global).squeeze(0)
        score_tor = self.tor_head(joint_global).squeeze(0)

        # ---- 侧链 chi1 分数 (DiffDock-Pocket 新增) ----
        score_sc = self.sc_tor_head(prot_h)             # (N_r, 1)

        return score_tr, score_rot, score_tor, score_sc

In [ ]:
# ============================================================
# 实例化模型并展示参数量
# ============================================================

model = ToyDiffDockPocketScore(
    lig_dim=ATOM_FEAT_DIM, res_dim=RES_FEAT_DIM, hidden_dim=HIDDEN_DIM
).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# 按模块统计参数
module_params = []
for name, module in model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    module_params.append({'模块': name, '参数量': f'{n_params:,}'})

params_summary = pd.DataFrame(module_params)
display(params_summary)

print(f"\n总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")

## 5. 训练

训练采用 **4-way 分数匹配 (score matching)** 策略:

1. 对每个样本，随机采样时间步 $t \in (0, 1]$
2. 对配体施加 3-way 噪声 (平移 + 旋转 + 扭转)
3. 对侧链 chi1 角施加噪声 (**第 4 维!**)
4. 模型预测 4 个分数，与分数匹配目标 $\text{score} = -\text{noise} / \sigma^2$ 计算 MSE 损失
5. 总损失 = loss_tr + loss_rot + loss_tor + loss_sc

In [ ]:
# ============================================================
# 训练循环 -- 4-way 分数匹配
# ============================================================

print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"开始训练 {N_EPOCHS} 轮 (4-way score matching)...\n")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_losses = []
    loss_components = {'tr': [], 'rot': [], 'tor': [], 'sc': []}

    for item in train_loader:
        lig_feats = item['lig_feats']
        lig_coords = item['lig_coords']
        res_feats = item['res_feats']
        ca_coords = item['ca_coords']
        chi1_angles = item['chi1_angles']

        # ---- 采样随机时间步 t in (0, 1] ----
        t = np.random.uniform(0.01, 1.0)

        # ---- 1. 对配体施加 3-way 噪声 ----
        noisy_lig, noise_tr, noise_rot, noise_tor = apply_noise_to_ligand(lig_coords, t)

        # ---- 2. 对侧链 chi1 角施加噪声 (第 4 维!) ----
        noisy_chi1, noise_sc = apply_noise_to_sidechain(chi1_angles, t)

        # ---- 3. 构建交互图 (使用加噪坐标) ----
        dist_mat = distance_matrix(noisy_lig, ca_coords)
        lig_idx, prot_idx = np.where(dist_mat < DISTANCE_CUTOFF)
        if len(lig_idx) == 0:
            continue

        edge_index = np.stack([lig_idx, prot_idx], axis=0)
        edge_dist = dist_mat[lig_idx, prot_idx].reshape(-1, 1)

        # ---- 转换为张量 ----
        lig_f = torch.FloatTensor(lig_feats).to(DEVICE)
        res_f = torch.FloatTensor(res_feats).to(DEVICE)
        ei = torch.LongTensor(edge_index).to(DEVICE)
        ed = torch.FloatTensor(edge_dist).to(DEVICE)
        t_tensor = torch.FloatTensor([t]).to(DEVICE)

        # ---- 前向传播: 预测 4 个分数 ----
        score_tr, score_rot, score_tor, score_sc = model(lig_f, res_f, ei, ed, t_tensor)

        # ---- 4-way 分数匹配损失 ----
        # 分数匹配目标: score = -noise / sigma^2
        sigma_tr, sigma_rot, sigma_tor, sigma_sc = t_to_sigma_4way(t)

        target_tr = torch.FloatTensor(-noise_tr / (sigma_tr ** 2 + 1e-8)).to(DEVICE)
        loss_tr = ((score_tr - target_tr) ** 2).sum()

        target_rot = torch.FloatTensor(-noise_rot / (sigma_rot ** 2 + 1e-8)).to(DEVICE)
        loss_rot = ((score_rot - target_rot) ** 2).sum()

        target_tor = torch.FloatTensor([-noise_tor / (sigma_tor ** 2 + 1e-8)]).to(DEVICE)
        loss_tor = ((score_tor - target_tor) ** 2).sum()

        # 侧链 chi1 损失 -- DiffDock-Pocket 新增!
        target_sc = torch.FloatTensor(
            -noise_sc / (sigma_sc ** 2 + 1e-8)
        ).reshape(-1, 1).to(DEVICE)
        loss_sc = ((score_sc - target_sc) ** 2).mean()

        # ---- 总损失: 4 个分量之和 ----
        total_loss = loss_tr + loss_rot + loss_tor + loss_sc

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        train_losses.append(total_loss.item())
        loss_components['tr'].append(loss_tr.item())
        loss_components['rot'].append(loss_rot.item())
        loss_components['tor'].append(loss_tor.item())
        loss_components['sc'].append(loss_sc.item())

    # ---- 每 20 轮打印一次 ----
    if epoch % 20 == 0 or epoch == 1:
        avg_total = np.mean(train_losses) if train_losses else float('nan')
        avg_tr = np.mean(loss_components['tr']) if loss_components['tr'] else 0
        avg_rot = np.mean(loss_components['rot']) if loss_components['rot'] else 0
        avg_tor = np.mean(loss_components['tor']) if loss_components['tor'] else 0
        avg_sc = np.mean(loss_components['sc']) if loss_components['sc'] else 0
        print(f"  Epoch {epoch:>4d}/{N_EPOCHS}  |  Total: {avg_total:.4f}  "
              f"(tr={avg_tr:.3f}, rot={avg_rot:.3f}, tor={avg_tor:.3f}, sc={avg_sc:.3f})")

## 6. 评估与可视化

在测试集上执行 **4-way 反向扩散对接**:

1. 从纯噪声 ($t=1.0$) 开始，同时初始化配体坐标和侧链 chi1 角度
2. 经过 20 步反向扩散，逐步去噪:
   - 平移更新、旋转更新、扭转角更新 (与 DiffDock 相同)
   - **侧链 chi1 更新** (第 4 维去噪，DiffDock-Pocket 新增)
3. 评估指标:
   - **配体 RMSD** -- 预测位姿与真实位姿的均方根偏差
   - **侧链 chi1 RMSD** -- 预测侧链角度与真实角度的均方根偏差 (考虑周期性)

In [ ]:
# ============================================================
# 测试集评估 -- 4-way 反向扩散对接
# ============================================================

print("在测试集上评估 (反向扩散对接)...")

model.eval()
lig_rmsds = []
sc_rmsds = []
pdbids_list = []

# ---- 扩散采样参数 ----
N_STEPS = 20           # 反向扩散步数
DT = 1.0 / N_STEPS

with torch.no_grad():
    for item in test_loader:
        lig_feats = item['lig_feats']
        lig_coords_true = item['lig_coords']
        res_feats = item['res_feats']
        ca_coords = item['ca_coords']
        chi1_true = item['chi1_angles']
        pdbid = item['pdbid']

        # ---- 初始化: 从纯噪声开始 (t=1.0) ----
        noisy_lig, _, _, _ = apply_noise_to_ligand(lig_coords_true, 1.0)
        noisy_chi1, _ = apply_noise_to_sidechain(chi1_true, 1.0)

        # ---- 反向扩散: t=1.0 -> t=0 ----
        for step in range(N_STEPS):
            t = 1.0 - step * DT
            sigma_tr, sigma_rot, sigma_tor, sigma_sc = t_to_sigma_4way(t)

            # 构建交互图
            dist_mat = distance_matrix(noisy_lig, ca_coords)
            lig_idx, prot_idx = np.where(dist_mat < DISTANCE_CUTOFF)
            if len(lig_idx) == 0:
                break

            edge_index = np.stack([lig_idx, prot_idx], axis=0)
            edge_dist = dist_mat[lig_idx, prot_idx].reshape(-1, 1)

            lig_f = torch.FloatTensor(lig_feats).to(DEVICE)
            res_f = torch.FloatTensor(res_feats).to(DEVICE)
            ei = torch.LongTensor(edge_index).to(DEVICE)
            ed = torch.FloatTensor(edge_dist).to(DEVICE)
            t_tensor = torch.FloatTensor([t]).to(DEVICE)

            score_tr, score_rot, score_tor, score_sc = model(
                lig_f, res_f, ei, ed, t_tensor
            )

            # ---- 去噪更新 (朗之万动力学) ----
            step_size = DT

            # 1. 平移更新
            noisy_lig = noisy_lig + score_tr.cpu().numpy() * sigma_tr ** 2 * step_size

            # 2. 旋转更新
            rot_update = score_rot.cpu().numpy() * sigma_rot ** 2 * step_size
            angle = np.linalg.norm(rot_update)
            if angle > 1e-6:
                axis = rot_update / angle
                center = noisy_lig.mean(axis=0)
                cos_a, sin_a = np.cos(angle), np.sin(angle)
                centered = noisy_lig - center
                cross = np.cross(axis, centered)
                dot = np.dot(centered, axis).reshape(-1, 1)
                noisy_lig = (centered * cos_a + cross * sin_a
                             + axis * dot * (1 - cos_a) + center)

            # 3. 扭转角更新
            tor_update = score_tor.cpu().numpy().item() * sigma_tor ** 2 * step_size
            center = noisy_lig.mean(axis=0)
            radial = noisy_lig - center
            radial_norms = np.linalg.norm(radial, axis=1, keepdims=True)
            radial_norms = np.clip(radial_norms, 1e-6, None)
            noisy_lig = noisy_lig + (radial / radial_norms) * tor_update * 0.1

            # 4. 侧链 chi1 更新 (第 4 维去噪!)
            sc_update = score_sc.cpu().numpy().flatten() * sigma_sc ** 2 * step_size
            noisy_chi1 = noisy_chi1 + sc_update
            noisy_chi1 = np.arctan2(np.sin(noisy_chi1), np.cos(noisy_chi1))

        # ---- 计算配体 RMSD (质心对齐后) ----
        pred_center = noisy_lig.mean(axis=0)
        true_center = lig_coords_true.mean(axis=0)
        pred_aligned = noisy_lig - pred_center + true_center
        lig_rmsd = compute_rmsd(pred_aligned, lig_coords_true)
        lig_rmsds.append(lig_rmsd)

        # ---- 计算侧链 chi1 角度 RMSD (考虑周期性) ----
        chi1_diff = noisy_chi1 - chi1_true
        chi1_diff = np.arctan2(np.sin(chi1_diff), np.cos(chi1_diff))
        sc_rmsd = np.sqrt(np.mean(chi1_diff ** 2))
        sc_rmsds.append(sc_rmsd)

        pdbids_list.append(pdbid)

lig_rmsds = np.array(lig_rmsds)
sc_rmsds = np.array(sc_rmsds)

In [ ]:
# ============================================================
# 评估结果展示与可视化
# ============================================================

# ---- 评估结果表格 ----
results_df = pd.DataFrame({
    '指标': [
        '样本数',
        '配体 RMSD (mean)',
        '配体 RMSD (median)',
        '侧链 chi1 RMSD (mean)',
        '侧链 chi1 RMSD (median)',
        '配体 RMSD < 2A',
        '配体 RMSD < 5A',
    ],
    '值': [
        f'{len(lig_rmsds)}',
        f'{lig_rmsds.mean():.4f} A',
        f'{np.median(lig_rmsds):.4f} A',
        f'{np.degrees(sc_rmsds.mean()):.2f} deg',
        f'{np.degrees(np.median(sc_rmsds)):.2f} deg',
        f'{(lig_rmsds < 2.0).sum()}/{len(lig_rmsds)} ({100.0*(lig_rmsds < 2.0).mean():.1f}%)',
        f'{(lig_rmsds < 5.0).sum()}/{len(lig_rmsds)} ({100.0*(lig_rmsds < 5.0).mean():.1f}%)',
    ]
})
display(results_df)

# ---- 可视化: 配体 RMSD 分布 + 侧链 chi1 RMSD ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 子图 1: 配体 RMSD 直方图
ax1 = axes[0]
ax1.hist(lig_rmsds, bins=20, color='steelblue', edgecolor='black', alpha=0.8)
ax1.axvline(x=2.0, color='red', linestyle='--', linewidth=1.5, label='2 A threshold')
ax1.axvline(x=5.0, color='orange', linestyle='--', linewidth=1.5, label='5 A threshold')
ax1.set_xlabel('Ligand RMSD (A)', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title(f'Ligand RMSD Distribution\n'
              f'Mean={lig_rmsds.mean():.2f} A, Median={np.median(lig_rmsds):.2f} A',
              fontsize=12)
ax1.legend(fontsize=10)

# 子图 2: 侧链 chi1 RMSD vs 配体 RMSD 散点图
ax2 = axes[1]
ax2.scatter(lig_rmsds, np.degrees(sc_rmsds), alpha=0.6, edgecolors='k',
            linewidths=0.5, s=40, color='coral')
ax2.set_xlabel('Ligand RMSD (A)', fontsize=12)
ax2.set_ylabel('Sidechain chi1 RMSD (deg)', fontsize=12)
ax2.set_title(f'DiffDock-Pocket: 4-way Diffusion\n'
              f'Sidechain RMSD Mean={np.degrees(sc_rmsds.mean()):.1f} deg',
              fontsize=12)
ax2.annotate('4th diffusion dim:\nsidechain chi1',
             xy=(0.95, 0.95), xycoords='axes fraction',
             ha='right', va='top', fontsize=9,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

## 总结

本教程实现了 DiffDock-Pocket 的教学版本，重点展示了其相比 DiffDock 的核心创新:

### 关键知识点

1. **4-way 扩散**: 在 DiffDock 的配体平移、旋转、扭转角 3 维扩散基础上，新增蛋白侧链 chi1 扭转角作为第 4 个扩散维度
2. **残基级别蛋白表示**: 使用 CA/CB 坐标和氨基酸类型独热编码，相比全原子表示更高效
3. **双向消息传递**: 蛋白残基和配体原子之间的交互信息在交互层中双向传递
4. **4 个分数头**: tr_head + rot_head + tor_head + sc_tor_head，其中 sc_tor_head 为每个残基独立预测侧链扭转分数
5. **分数匹配训练**: 对 4 个扩散维度分别计算分数匹配损失，总损失为四项之和
6. **反向扩散推理**: 从纯噪声出发，通过朗之万动力学同时优化配体位姿和蛋白口袋构象

### 教学简化说明

- 扭转角噪声使用径向扰动代替真实的可旋转键操作
- chi1 角度使用 CA->CB 方向的投影角作为简化代理
- 训练数据为 CASF-2016 核心集 (285 个复合物)，实际应用需要更大规模的数据
- 实际模型使用更复杂的 SE(3) 等变网络架构